# Plot Back-slip inversion of the GNSS displacement measurements of interseismic locking (coupling) at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution



In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/real/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
# flag_savefig = True
flag_savefig = False

# Define the inversion results path
resultpath = "rst_coseis/"

# define mesh name
# meshname = "nicoya"
# meshname = "nicoya2"   # This has a smaller fault interface
meshname = "nicoya3"   # the same as above but 5-km mesh size on fault
# meshname = "nicoya4"   # extended fault area
print(meshname)

# Read the plate interface file
plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
df_plate = utp.parse_plate_interface_file(datadir + plate_file)
depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the plate file in GMT grd format
grd_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"

# Read the trench file
trench_file = "plateinterface/trench.gmt.inuse"

# read in the lon and lat of stations
obs_file = "data/Protti_etal_2014_tableS1.csv"
# note that the height is in m, duration and dates are in yr, and the displacements and errors are already in m
# From BAGA to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                   names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                          'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

# displacement noise standard deviations, in m 
error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())


In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
# print(volcano.head())

# # region for plotting
# region1=[-87+360, -84+360, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# # region1=[-86.75+360, -84.4+360, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88+360, -83+360, 6, 14]    # suitable region for chopping the plate interface grid file 

# # Filter by both longitude and latitude
# segments = utp.parse_trench_file(datadir + trench_file, 
#                            lon_range=(region[0]-360, region[1]-360), 
#                            lat_range=(region[2], region[3]))
# # Convert to DataFrame for analysis
# trench = utp.segments_to_dataframe(segments)

# volcano = volcano[
#     (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
#     (volcano['lon'] >= region[0]-360) & (volcano['lon'] <= region[1]-360)
# ]

# region for plotting
# region1=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

# Filter by both longitude and latitude
segments = utp.parse_trench_file(datadir + trench_file, 
                           lon_range=(region[0], region[1]), 
                           lat_range=(region[2], region[3]))
# Convert to DataFrame for analysis
trench = utp.segments_to_dataframe(segments)
print(trench.head())

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]


In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1     # same as in coseismic case

# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# the optimal weight combination comes from the L-curve test
# rho_s = 1e7
# rho_s = 1e8
rho_s = 1e9

# gamma_val_H1 = 1e2
gamma_val_H1 = 4e2
# gamma_val_H1 = 1e3

delta_val_L2 = gamma_val_H1 / rho_s 

# preferred damping for unconstrained inversion on the extended fault
if meshname == "nicoya3": 
    gamma_val_H1 = 1e2  
    # delta_val_L2 = 2.5e-6
    delta_val_L2 = 1e-5
elif meshname == "nicoya4":   # extended fault
    gamma_val_H1 = 1e2  
    # delta_val_L2 = 5e-6
    delta_val_L2 = 7.5e-6
    
inv_str = f"_coseis_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"

print(meshname)
print(inv_str)

In [ ]:
# Load the ground-truth displacement for each forward structure model
def read_obs_disp(datadir, resultpath, meshname, inv_str):
    outFileName = 'd_obs_' + meshname + inv_str + '.txt'
    print(outFileName)
    u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])    
    # print(u_obs.head())

    return u_obs

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(df, u_obs, error_e, error_n, error_v):
    # convert from m to mm, horizontal components
    df_obs_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_obs['ux']*1000,
            "north_velocity": u_obs['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    # print(df_obs_h.head())
    # np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

    # convert from m to mm, vertical components
    df_obs_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_obs['uz']*1000,
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    
    return df_obs_h, df_obs_v

In [ ]:
# read in the observation disp.
u_obs = read_obs_disp(datadir, resultpath, meshname, inv_str)

# buil observation disp. vectors
df_obs_h, df_obs_v = build_disp_vector(df, u_obs, error_e, error_n, error_v)

In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
# mu_l = -0.9730  # ~25 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = 0.9730 # ~55 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f" %(mu_upper, mu_lower) )
    # string for the contrast between slab and wedge
    slabwedge_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the 3d model
    het3d_str = "_DeShon3D"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    anomaly_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(slabwedge_str)
print(het3d_str)

# # define the structure model strings for the INVERSION 
# struc_str_inv_het = anomaly_str
# print("Inverse problem based on: ", struc_str_inv_het)

# struc_str_inv_hom = homo_str
# print("Inverse problem based on: ", struc_str_inv_hom)


In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -85.5+360, 10
R = 6371.0  # Earth radius in km
km2m = 1e3  # conversion factor from km to m

# Load the fault geometry, all in meters
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f/km2m

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = utp.inverse_azi_equidist_proj(loc_f['xf'], loc_f['yf'], lon0, lat0)
print(loc_f[['lon', 'lat']].head())


In [ ]:
# Load the predicted surface displacement, all in meters
def read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    u_res = u_pred.copy()
    u_res['ux'] = u_obs['ux'] - u_pred['ux']
    u_res['uy'] = u_obs['uy'] - u_pred['uy']
    u_res['uz'] = u_obs['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    # u_res.head()

    return u_pred, u_res

# Load the inferred slip on the fault, all in meters
def read_inferred_slip(datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])
    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)

    # print(m_s['s_strike'].min(), m_s['s_strike'].max())
    # print(m_s['s_dip'].min(), m_s['s_dip'].max())
    # print(m_s['mag'].min(), m_s['mag'].max())
    print(m_s['mag'].max())

    return m_s

# Load the inferred moment and potency of the slip
def read_moment_potency(datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'moment_' + meshname + inv_str + struc_str_inv + '.txt'
    # print(outFileName)
    
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['moment', 'Mw', 'potency'])
    print(mom.head())

    # print("Inferred moment (Nm): ", mom['moment'].values[0])
    # print("Inferred moment magnitude Mw: ", mom['Mw'].values[0])
    # print("Inferred potency (m^3): ", mom['potency'].values[0])

    return mom

In [ ]:
##### Load the results of the homogeneous inversion
# Load the predicted surface displacement, all in meters
u_pred_hom, u_res_hom = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, homo_str)
# Load the inferred slip on the fault, all in meters
m_s_hom = read_inferred_slip(datadir, resultpath, meshname, inv_str, homo_str)
mom_hom = read_moment_potency(datadir, resultpath, meshname, inv_str, homo_str)

##### Load the results of the slab/wedge model inversion
u_pred_sw, u_res_sw = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, slabwedge_str)
m_s_sw = read_inferred_slip(datadir, resultpath, meshname, inv_str, slabwedge_str)
mom_sw = read_moment_potency(datadir, resultpath, meshname, inv_str, slabwedge_str)

##### Load the results of the 3d model inversion
u_pred_3d, u_res_3d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het3d_str)
m_s_3d = read_inferred_slip(datadir, resultpath, meshname, inv_str, het3d_str)
mom_3d = read_moment_potency(datadir, resultpath, meshname, inv_str, het3d_str)

# difference between the heterogeneous and homogeneous model
print((m_s_sw['mag']-m_s_hom['mag']).min(), (m_s_sw['mag']-m_s_hom['mag']).max())
print((m_s_3d['mag']-m_s_hom['mag']).min(), (m_s_3d['mag']-m_s_hom['mag']).max())


In [ ]:
def plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.1c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            maxslip = max(m_s_hom[col2plt].max(), m_s_sw[col2plt].max(), m_s_3d[col2plt].max())
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_hom[col2plt].max():.1f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_hom['potency'].iloc[0]:.1e} m^3", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")          
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the slab/wedge model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+tS/W inv."])
            # maxslip = m_s_sw[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_sw[col2plt].max():.1f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_sw['potency'].iloc[0]:.1e} m^3", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the 3d model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+t3D inv."])
            # maxslip = m_s_3d[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. slip: {m_s_3d[col2plt].max():.1f} m", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"Potency: {mom_3d['potency'].iloc[0]:.1e} m^3", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_sum_invslip.pdf")

# plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, 'locking', region, flag_savefig=True)
# plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, 'locking2', region, flag_savefig=False)
# plot_all_inferred_slip_(m_s_hom, m_s_hom, m_s_hom, 'mag', region, flag_savefig=False)
plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, 'mag', region1, flag_savefig=False)


In [ ]:
def plot_homvshet_slip(m_s_hom, m_s, col2plt, region, struc_str_inv, flag_savefig=False):

    slipsybsz = "c0.1c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = 1
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # slip inferred from the heterogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet. inv."])
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(m)"]) #

        # difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet - Hom"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference", "y+l(m)"]) #

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_hethomslip.pdf")


plot_homvshet_slip(m_s_hom, m_s_sw, 'mag', region1, slabwedge_str, flag_savefig=False)    
plot_homvshet_slip(m_s_hom, m_s_3d, 'mag', region1, het3d_str, flag_savefig=False)    

In [ ]:
# plot the histogram of difference in slip
fig, ax = plt.subplots(figsize=(4, 4.5), dpi=300)

xmin, xmax = -0.25, 0.25
bin_width = 0.01
bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
median = np.median(m_s_mag['diff'])
sigma = np.std(m_s_mag['diff'])
# rounded_sigma = round(sigma)
counts1, _, _ = ax.hist(m_s_mag['diff'], bins=bins, color='dimgray', alpha=0.7, edgecolor='black')
ymin, ymax = 0, np.max(counts1) + 100
ax.errorbar(median, ymax*0.82, xerr=sigma, fmt='o', color='black', capsize=5, capthick=2, 
            linewidth=2, label=fr'$\pm1\sigma = \pm{sigma:.3f}$')
ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median = {median:.3f}')
ax.legend()
ax.set_xlabel('Difference (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Difference in magnitude (Het-Hom)', fontweight='bold')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)


In [ ]:
# for obs, pred vectors
m2cm = 1e2  # conversion factor from meters to centimeters
ref_cm = 20  # reference length in cm

# for error vectors
m2mm = 1e3  # conversion factor from meters to millimeters
ref_err_mm = 5   # reference length for error in mm

ref_rat = 10 # reference ratio for data and error vectors
# ref_rat = ref_cm / ref_err_cm  # ratio of reference lengths for data and error vectors

# A different way of constructing the vectors for plotting arrows
# convert from m to mm, horizontal components
df_obs_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_obs['ux']*m2cm/ref_cm,
        "north_velocity": u_obs['uy']*m2cm/ref_cm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)
df_obs_h.head()

np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

# vertical components
df_obs_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_obs['uz']*m2cm/ref_cm,
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)


In [ ]:
### data fitting by the heterogeneous inversion 
# convert from m to mm
df_pred_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_pred['ux']*m2cm/ref_cm,
        "north_velocity": u_pred['uy']*m2cm/ref_cm,        
    }
)
df_pred_h.head()

df_pred_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_pred['uz']*m2cm/ref_cm,        
    }
)

# for error vectors
ref_res_mm = 25   # reference length for error in mm
ref_err_mm = 5   # reference length for error in mm
ref_rat_res = 5 # reference ratio for data and error vectors

# convert from m to mm, in order to directly compare with 
df_res_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_res['ux']*m2mm/ref_res_mm,
        "north_velocity": u_res['uy']*m2mm/ref_res_mm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,        
    }
)
df_res_h.head()
df_res_h_dum = df_res_h.iloc[:, :4]

df_res_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_res['uz']*m2mm/ref_res_mm,    
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,     
    }
)
df_res_v_dum = df_res_v.iloc[:, :4]


In [ ]:
# plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_h, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat_res, "north_sigma": 1/ref_rat_res, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")

fig.show()


In [ ]:
### data fitting by the homogeneous inversion 
# convert from m to mm
df_pred_hom_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_pred_hom['ux']*m2cm/ref_cm,
        "north_velocity": u_pred_hom['uy']*m2cm/ref_cm,       
    }
)
df_pred_h.head()

df_pred_hom_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_pred_hom['uz']*m2cm/ref_cm,      
    }
)

# convert from m to mm, in order to directly compare with 
df_res_hom_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_res_hom['ux']*m2mm/ref_res_mm,
        "north_velocity": u_res_hom['uy']*m2mm/ref_res_mm,
        "east_sigma": error_e*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_n*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,        
    }
)
df_res_hom_h.head()
df_res_hom_h_dum = df_res_hom_h.iloc[:, :4]

df_res_hom_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_res_hom['uz']*m2mm/ref_res_mm,   
        "east_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "north_sigma": error_v*m2mm/ref_err_mm/ref_rat_res,
        "correlation_EN": 0.0,     
    }
)
df_res_hom_v_dum = df_res_hom_v.iloc[:, :4]


In [ ]:
# plot the predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_h, pen="0.5p,blue", uncertaintyfill=None, line="0.25p,blue", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat_res, "north_sigma": 1/ref_rat_res, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")


fig.show()


In [ ]:
# Plot the comparison on horizontal components
fig = pygmt.Figure()
with fig.subplot(nrows=2, ncols=3, figsize=("18c", "12c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # homogeneous Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")        

    # heterogeneous Observed vs. predicted
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

    # residuals, hom vs het
    with fig.set_panel(panel=[0, 2]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_h_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        fig.velo(data=df_res_h_dum, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres_hom = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.6, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        lgdres = pd.DataFrame(data={"x": region[0]+0.05, "y": region[-2]+2.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

    ### Plot the comparison on vectical components
    # homogeneous Observed vs. predicted
    with fig.set_panel(panel=[1, 0]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_hom_v, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")      

    # heterogeneous Observed vs. predicted
    with fig.set_panel(panel=[1, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,black+ea+gblack")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_v, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/ref_rat, "north_sigma": 1/ref_rat, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+2.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+2.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_cm} cm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="8p,Helvetica,black", justify="ML")

    # residuals, hom vs het
    with fig.set_panel(panel=[1, 2]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_hom_v_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,blue+ea+gblue")
        fig.velo(data=df_res_v_dum, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+2.5, style="r2/1.7", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+2.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres_hom = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+2.4, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
        lgdres = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+2.2, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+2.8, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+2.6, font="6p,Helvetica,black", justify="ML")
        fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+2.4, font="6p,Helvetica,black", justify="ML")
        fig.text(text=f"{ref_res_mm} mm ± {ref_err_mm} mm", x=region[0]+0.1, y=region[-2]+2.2, font="6p,Helvetica,black", justify="ML")

fig.show()  

In [ ]:
# plot the histogram of difference in residuals
fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.5), dpi=300)

bin_width = 2

ax = axes[0]

xmin, xmax = 0, 50 
bins = np.arange(xmin, xmax + bin_width, bin_width)
counts1, _, _ = ax.hist(u_res_hom['mag']*1000, bins=bins, color='blue', alpha=0.7, edgecolor='black',label='Homogeneous')
counts2, _, _ = ax.hist(u_res['mag']*1000, bins=bins, color='red', alpha=0.7, edgecolor='black',label='Heterogeneous')

ymin, ymax = 0, max(np.max(counts1), np.max(counts2)) + 4

median_hom = np.median(u_res_hom['mag']*1000)
sigma_hom = np.std(u_res_hom['mag']*1000)
median = np.median(u_res['mag']*1000)
sigma = np.std(u_res['mag']*1000)

rmse_hom = ut.rmse_3d_dataframe(u_pred_hom, u_obs)* 1000  # Convert to mm
print(f"HOM Residual RMSE: {rmse_hom:.2f} mm")
print(f"HOM Residual median: {median_hom:.2f} mm, sigma: {sigma_hom:.2f} mm")

rmse = ut.rmse_3d_dataframe(u_pred, u_obs)* 1000  # Convert to mm
print(f"HET Residual RMSE: {rmse:.2f} mm")
print(f"HET Residual median: {median:.2f} mm, sigma: {sigma:.2f} mm")

ax.plot([median_hom, median_hom], [ymin, ymax], '--', color='blue', label=fr'Hom median={median_hom:.1f}')
ax.plot([median, median], [ymin, ymax], '--', color='red', label=fr'Het median={median:.1f}')
ax.errorbar(median_hom, ymax-2, xerr=sigma_hom, fmt='o', color='blue', capsize=5, capthick=2, 
            linewidth=2, label=fr'Hom $\pm1\sigma=\pm{sigma_hom:.1f}$')
ax.errorbar(median, ymax-2, xerr=sigma, fmt='o', color='red', capsize=5, capthick=2, 
            linewidth=2, label=fr'Het $\pm1\sigma=\pm{sigma:.1f}$')
ax.text(0.1, 0.9, f"Hom Res RMSE:{rmse_hom:.2f}", fontsize=9, color='k', transform=ax.transAxes)
ax.text(0.1, 0.8, f"Het Res RMSE:{rmse:.2f}", fontsize=9, color='k', transform=ax.transAxes)

ax.legend(fontsize=9, loc='best')
ax.set_xlabel('Residual (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax+2, 2))
ax.set_xlim(xmin, xmax)


ax = axes[1]
bin_width = 1
xmin, xmax = -15, 15
bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
counts1, _, _ = ax.hist(u_res['mag']*1000-u_res_hom['mag']*1000, bins=bins, color='dimgray',alpha=0.7, edgecolor='black',label='het-hom')
ymin, ymax = 0, np.max(counts1) + 2

median = np.median(u_res['mag']*1000-u_res_hom['mag']*1000)
ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median={median:.2f}')

ax.legend(fontsize=9, loc='right')
ax.set_xlabel('Difference (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Residual difference (Het-Hom)', fontweight='bold')
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax+2, 2))
ax.set_xlim(xmin, xmax)